In [1]:
import os
import shutil
from tqdm import tqdm

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9

# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# api.py
COPY api.py .

# app.py
COPY app.py .

# cls_parser.pkl
COPY cls_parser.pkl .

# datascience_rsa_key.p8
COPY datascience_rsa_key.p8 .

# functions_app
COPY functions_app.py .

# functions
COPY functions.py .

# functions_counters
COPY functions_counters.py .

# functions_counters
COPY functions_counters_pricing.py .

# preprocessing.py
COPY preprocessing.py .

# df_aa.csv
COPY df_descriptions.csv .

# static
COPY static ./static

# templates
COPY templates ./templates

# run script when image is run
CMD ["python3", "app.py"]

Writing Dockerfile


### Build and push to ECR

In [3]:
%%sh

# name the image
image=gen12-payload-comparison

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 791B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.9
#2 DONE 0.5s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [ 1/17] FROM docker.io/library/python:3.9@sha256:c17c71e1f5f258803a6b7c391f8013adbf84285af54c2a811de4a5a1ac5a8676
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 15.99kB done
#5 DONE 0.0s

#6 [ 4/17] COPY requirements.txt .
#6 CACHED

#7 [ 9/17] COPY datascience_rsa_key.p8 .
#7 CACHED

#8 [ 6/17] COPY api.py .
#8 CACHED

#9 [11/17] COPY functions.py .
#9 CACHED

#10 [ 2/17] RUN apt-get update
#10 CACHED

#11 [ 8/17] COPY cls_parser.pkl .
#11 CACHED

#12 [ 7/17] COPY app.py .
#12 CACHED

#13 [ 3/17] RUN pip install --upgrade pip
#13 CACHED

#14 [10/17] COPY functions_app.py .
#14 CACHED

#15 [ 5/17] RUN pip install -r requirements.txt
#15 CACHED

#16

Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'gen12-payload-comparison' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/gen12-payload-comparison]
f85a42c44377: Preparing
6e038482e4a1: Preparing
fabaab557f5c: Preparing
59a70923cdd6: Preparing
095eb01b6ed8: Preparing
a48d28826629: Preparing
778a58fbf85d: Preparing
76622d474c65: Preparing
ba847675134c: Preparing
72d2828165d5: Preparing
600dcb343e9a: Preparing
0761593429f2: Preparing
2822e5cd2d07: Preparing
3de6e5e85939: Preparing
a48d28826629: Waiting
eeeac0ae254b: Preparing
778a58fbf85d: Waiting
0d371be961b4: Preparing
76622d474c65: Waiting
60a159600b22: Preparing
ee959616fc20: Preparing
d0e85779261a: Preparing
dafb8aed9f7f: Preparing
41d4dc7516bb: Preparing
c0f51bbdc37d: Preparing
91b542912d12: Preparing
ba847675134c: Waiting
60a159600b22: Waiting
72d2828165d5: Waiting
ee959616fc20: Waiting
600dcb343e9a: Waiting
d0e85779261a: Waiting
dafb8aed9f7f: Waiting
0761593429f2: Waiting
41d4dc7516bb: Waiting
2822e5cd2d07: Waiting
c0f51bbdc37d: Waiting
91b542912d12: Waiting
3de6e5e85939: Wa

### Clean up Dockerfile

In [4]:
try:
    os.remove('Dockerfile')
except FileNotFoundError:
    pass